# Paper-style β-VAE adapted to PROPHESY C1s spectra

This notebook implements the fully connected β-VAE described in *Unsupervised real-world knowledge extraction via disentangled variational autoencoders for photon diagnostics*, adapted to the PROPHESY-generated Excel files.

The paper used `14000 → 2824 → 579 → 117 → 24 → 12`, where the 24 encoder outputs represent 12 means and 12 log-variances. PROPHESY C1s spectra contain 401 points, so the corresponding network used here is `401 → 256 → 128 → 64 → 24 → 12`, with the mirrored decoder `12 → 64 → 128 → 256 → 401`. The VAE architecture itself is unchanged from the previous notebook: twelve latent variables, fully connected layers, Mish activations, a sigmoid output, and the β-weighted KL objective.

Peak parameters are **not used for training**. They are retained only to test whether physical properties emerge in the latent space. PROPHESY's noise-free `Stot` is used as the reconstruction target while `Snoisy` remains the input, preventing the network from being rewarded for reproducing Poisson noise.

## Requirements

Use a Python environment containing PyTorch, NumPy, OpenPyXL, and Matplotlib. If necessary, install them in the active notebook kernel with:

```python
%pip install torch numpy openpyxl matplotlib
```

The cell is shown as documentation rather than executed automatically so the notebook does not modify your environment without permission.

In [ ]:
from pathlib import Path
import copy
import json
import math
import random
import time

import matplotlib.pyplot as plt
import numpy as np
from openpyxl import load_workbook
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__}; device={DEVICE}")

In [ ]:
# Paths work when Jupyter is started in spectraVAE. The absolute fallback matches
# the current shared workspace and can be changed if the projects are moved.
prophesy_candidates = [
    (Path.cwd() / "../PROPHESY").resolve(),
    Path("/home/umar/projects/Thesis/PROPHESY"),
]
PROPHESY_ROOT = next((p for p in prophesy_candidates if (p / "data").exists()), None)
if PROPHESY_ROOT is None:
    raise FileNotFoundError("Set PROPHESY_ROOT to the PROPHESY project directory.")

DATA_ROOT = PROPHESY_ROOT / "data"
CACHE_DIR = Path.cwd() / "data_cache"
OUTPUT_DIR = Path.cwd() / "outputs" / "prophesy_beta_vae"
CACHE_FILE = CACHE_DIR / "prophesy_c1s_401.npz"
CHECKPOINT_FILE = OUTPUT_DIR / "best_beta_vae.pt"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_LENGTH = 401
REBUILD_CACHE = False
print(f"PROPHESY data: {DATA_ROOT}")
print(f"Cache: {CACHE_FILE}")

## Extract compatible PROPHESY spectra

Each `Snoisy` or `Snoisy_*` column becomes one sample. The extractor keeps only non-water, non-S2p C1s sheets with 401 binding-energy points and a common energy axis. It also stores the simulated clean signal, background, expected total signal, photon energy, and three true peak components.

The workbook's parent directory becomes `case_id`. All spectra from the same case will remain in one train/validation/test partition, preventing repeated noise realizations of the same physical spectrum from leaking between partitions.

In [ ]:
def _column(headers, name):
    return headers.index(name) if name in headers else None


def extract_prophesy_c1s(data_root: Path, target_length: int = 401):
    noisy_parts, clean_parts, background_parts, total_parts = [], [], [], []
    centers_parts, widths_parts, fractions_parts = [], [], []
    photon_energy_parts, case_parts, source_parts, sweep_parts = [], [], [], []
    reference_axis = None
    skipped_axis = 0
    skipped_incomplete = 0

    workbook_paths = sorted(data_root.rglob("data.xlsx"))
    for workbook_path in workbook_paths:
        # Start with one chemically consistent family and one fixed grid.
        if "S2p" in workbook_path.parts or "water" in workbook_path.parts:
            continue

        case_id = str(workbook_path.parent.relative_to(data_root))
        workbook = load_workbook(workbook_path, read_only=True, data_only=True)
        sheet_names = set(workbook.sheetnames)

        for sheet_name in workbook.sheetnames:
            if not sheet_name.startswith("hν_"):
                continue
            ws = workbook[sheet_name]
            if ws.max_row - 1 != target_length:
                continue

            headers = [cell.value for cell in next(ws.iter_rows(min_row=1, max_row=1))]
            required = {"Be", "Snoisy", "SpectrumA_1", "Sbg", "Stot", "hν"}
            if not required.issubset(set(headers)):
                skipped_incomplete += 1
                continue

            rows = list(ws.iter_rows(min_row=2, values_only=True))
            block = np.asarray(rows, dtype=np.float64)
            axis = block[:, _column(headers, "Be")].astype(np.float32)
            if reference_axis is None:
                reference_axis = axis
            elif not np.allclose(axis, reference_axis, rtol=0.0, atol=1e-6):
                skipped_axis += 1
                continue

            noise_indices = [
                i for i, name in enumerate(headers)
                if name == "Snoisy" or (isinstance(name, str) and name.startswith("Snoisy_"))
            ]
            if not noise_indices:
                continue

            noisy = block[:, noise_indices].T.astype(np.float32)
            n_samples = noisy.shape[0]
            clean = block[:, _column(headers, "SpectrumA_1")].astype(np.float32)
            background = block[:, _column(headers, "Sbg")].astype(np.float32)
            expected_total = block[:, _column(headers, "Stot")].astype(np.float32)
            photon_energy = float(block[0, _column(headers, "hν")])

            meta_name = f"meta_{sheet_name}"
            if meta_name not in sheet_names:
                skipped_incomplete += 1
                continue
            meta_ws = workbook[meta_name]
            meta_headers = [cell.value for cell in next(meta_ws.iter_rows(min_row=1, max_row=1))]
            meta_rows = np.asarray(
                list(meta_ws.iter_rows(min_row=2, values_only=True)), dtype=np.float64
            )
            if meta_rows.ndim != 2 or meta_rows.shape[0] < 3:
                skipped_incomplete += 1
                continue
            centers = meta_rows[:3, _column(meta_headers, "peak_mode")].astype(np.float32)
            widths = meta_rows[:3, _column(meta_headers, "peak_width")].astype(np.float32)
            fractions = meta_rows[:3, _column(meta_headers, "peak_probability")].astype(np.float32)

            noisy_parts.append(noisy)
            clean_parts.append(np.repeat(clean[None, :], n_samples, axis=0))
            background_parts.append(np.repeat(background[None, :], n_samples, axis=0))
            total_parts.append(np.repeat(expected_total[None, :], n_samples, axis=0))
            centers_parts.append(np.repeat(centers[None, :], n_samples, axis=0))
            widths_parts.append(np.repeat(widths[None, :], n_samples, axis=0))
            fractions_parts.append(np.repeat(fractions[None, :], n_samples, axis=0))
            photon_energy_parts.append(np.full(n_samples, photon_energy, dtype=np.float32))
            case_parts.extend([case_id] * n_samples)
            source_parts.extend([f"{workbook_path.relative_to(data_root)}::{sheet_name}"] * n_samples)
            sweep_parts.extend([str(headers[i]) for i in noise_indices])

        workbook.close()

    if not noisy_parts:
        raise RuntimeError("No compatible 401-point C1s spectra were found.")

    result = {
        "axis": reference_axis,
        "noisy": np.concatenate(noisy_parts),
        "clean": np.concatenate(clean_parts),
        "background": np.concatenate(background_parts),
        "expected_total": np.concatenate(total_parts),
        "peak_center": np.concatenate(centers_parts),
        "peak_width": np.concatenate(widths_parts),
        "peak_fraction": np.concatenate(fractions_parts),
        "photon_energy": np.concatenate(photon_energy_parts),
        "case_id": np.asarray(case_parts),
        "source": np.asarray(source_parts),
        "sweep": np.asarray(sweep_parts),
    }
    print(f"Skipped {skipped_axis} sheets with a different axis.")
    print(f"Skipped {skipped_incomplete} incomplete sheets.")
    return result

In [ ]:
if REBUILD_CACHE or not CACHE_FILE.exists():
    dataset = extract_prophesy_c1s(DATA_ROOT, TARGET_LENGTH)
    np.savez_compressed(CACHE_FILE, **dataset)
    print(f"Saved {CACHE_FILE}")
else:
    with np.load(CACHE_FILE, allow_pickle=False) as cached:
        dataset = {name: cached[name] for name in cached.files}
    print(f"Loaded {CACHE_FILE}")

print(f"Spectra: {dataset['noisy'].shape}")
print(f"Independent cases: {np.unique(dataset['case_id']).size}")
print(f"Photon-energy range: {dataset['photon_energy'].min():.1f}–{dataset['photon_energy'].max():.1f} eV")
print(f"Count range: {dataset['noisy'].min():.1f}–{dataset['noisy'].max():.1f}")

## Leakage-safe split and normalization

Splitting is performed by physical case rather than spectrum. Repeated noisy acquisitions from one simulated condition therefore cannot appear in both training and testing. Candidate group splits are scored for balance in photon energy, intensity, peak position, and peak width so that validation and test sets are not accidentally dominated by unusually easy or difficult cases.

PROPHESY stores Poisson-like non-negative counts with a much wider dynamic range than the paper's 8-bit ADC values. The notebook therefore uses the variance-stabilizing Anscombe transform `2√(x + 3/8)`, scaled from training data into `[0, 1]`. Its inverse is quadratic rather than exponential, so small decoder errors are no longer magnified into extreme raw-count errors. Training batches use inverse-frequency sampling by physical spectrum (`source`) so fifty repeated sweeps do not give one configuration fifty times more influence than a single-sweep configuration.

In [ ]:
all_cases = np.unique(dataset["case_id"])
case_ids = dataset["case_id"]
n_cases = len(all_cases)
n_test = max(1, int(round(0.15 * n_cases)))
n_val = max(1, int(round(0.15 * n_cases)))

# Describe each physical case with variables that previously differed strongly across splits.
case_features = []
for case in all_cases:
    idx = np.flatnonzero(case_ids == case)
    case_features.append([
        np.mean(dataset["photon_energy"][idx]),
        np.mean(np.log1p(dataset["expected_total"][idx].sum(axis=1))),
        np.mean(dataset["peak_center"][idx]),
        np.mean(dataset["peak_width"][idx]),
    ])
case_features = np.asarray(case_features, dtype=np.float64)
case_features = (case_features - case_features.mean(axis=0)) / np.maximum(case_features.std(axis=0), 1e-8)

# Search reproducibly for a group split whose validation/test summaries resemble all cases.
rng = np.random.default_rng(SEED)
best_score, best_partition = np.inf, None
for _ in range(2000):
    order = rng.permutation(n_cases)
    test_case_idx = order[:n_test]
    val_case_idx = order[n_test:n_test + n_val]
    score = (
        np.mean(np.abs(case_features[test_case_idx].mean(axis=0)))
        + np.mean(np.abs(case_features[val_case_idx].mean(axis=0)))
    )
    if score < best_score:
        best_score = score
        best_partition = (test_case_idx.copy(), val_case_idx.copy(), order[n_test + n_val:].copy())
test_case_idx, val_case_idx, train_case_idx = best_partition
test_cases = set(all_cases[test_case_idx])
val_cases = set(all_cases[val_case_idx])
train_cases = set(all_cases[train_case_idx])
train_idx = np.flatnonzero(np.isin(case_ids, list(train_cases)))
val_idx = np.flatnonzero(np.isin(case_ids, list(val_cases)))
test_idx = np.flatnonzero(np.isin(case_ids, list(test_cases)))
assert train_cases.isdisjoint(val_cases | test_cases)
assert val_cases.isdisjoint(test_cases)

def anscombe(x):
    return 2.0 * np.sqrt(np.maximum(x, 0.0) + 3.0 / 8.0)

# Five percent headroom prevents saturation at the largest training count.
TRANSFORM_SCALE = 1.05 * float(anscombe(dataset["expected_total"][train_idx]).max())
if not np.isfinite(TRANSFORM_SCALE) or TRANSFORM_SCALE <= 0:
    raise ValueError("Invalid Anscombe normalization scale.")

def normalize_counts(x):
    return np.clip(anscombe(x) / TRANSFORM_SCALE, 0.0, 1.0).astype(np.float32)

def denormalize_counts(x):
    transformed = np.clip(x, 0.0, 1.0) * TRANSFORM_SCALE
    return np.maximum((transformed / 2.0) ** 2 - 3.0 / 8.0, 0.0)

x_all = normalize_counts(dataset["noisy"])
target_all = normalize_counts(dataset["expected_total"])

def make_loader(indices, training=False):
    tensors = TensorDataset(torch.from_numpy(x_all[indices]), torch.from_numpy(target_all[indices]))
    sampler = None
    if training:
        sources = dataset["source"][indices]
        unique_sources, source_counts = np.unique(sources, return_counts=True)
        count_by_source = dict(zip(unique_sources.tolist(), source_counts.tolist()))
        weights = torch.as_tensor([1.0 / count_by_source[s] for s in sources], dtype=torch.double)
        sampler = WeightedRandomSampler(weights, num_samples=len(indices), replacement=True)
    return DataLoader(
        tensors, batch_size=256, shuffle=False, sampler=sampler, num_workers=0,
        pin_memory=torch.cuda.is_available(), drop_last=False,
    )

train_loader = make_loader(train_idx, training=True)
val_loader = make_loader(val_idx)
test_loader = make_loader(test_idx)
print(f"train={len(train_idx):,}, validation={len(val_idx):,}, test={len(test_idx):,}")
print(f"train cases={len(train_cases)}, validation cases={len(val_cases)}, test cases={len(test_cases)}")
print(f"balanced group-split score={best_score:.4f}")
print(f"Anscombe normalization scale={TRANSFORM_SCALE:.4f}")

## β-VAE architecture

The final encoder layer produces 24 values, split into twelve-dimensional `mu` and `logvar`. The decoder mirrors the encoder but excludes that distribution-parameter layer, as in the paper.

In [ ]:
class BetaVAE(nn.Module):
    def __init__(self, input_dim=401, latent_dim=12):
        super().__init__()
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256), nn.Mish(),
            nn.Linear(256, 128), nn.Mish(),
            nn.Linear(128, 64), nn.Mish(),
        )
        self.latent_statistics = nn.Linear(64, 2 * latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64), nn.Mish(),
            nn.Linear(64, 128), nn.Mish(),
            nn.Linear(128, 256), nn.Mish(),
            nn.Linear(256, input_dim), nn.Sigmoid(),
        )
        self.apply(self._initialize)

    @staticmethod
    def _initialize(module):
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            nn.init.zeros_(module.bias)

    def encode(self, x):
        statistics = self.latent_statistics(self.encoder(x))
        return statistics.chunk(2, dim=-1)

    @staticmethod
    def reparameterize(mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

model = BetaVAE(input_dim=TARGET_LENGTH, latent_dim=12).to(DEVICE)
print(model)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## Training

The VAE architecture remains the paper-style network above. The reconstruction objective is adapted to the known PROPHESY noise model: noisy counts are encoded, while the decoder is compared with the noise-free expected spectrum. Pointwise Anscombe-space MSE is supplemented with first- and second-difference consistency, integrated-area error, and peak-height error. These terms correct peak shape and amplitude without adding supervised peak labels or changing the network. β is warmed from zero to the paper's `0.034`, allowing reconstruction to form before enforcing the normal prior. Adam starts at `3×10⁻⁴`; `ReduceLROnPlateau` lowers it only when validation genuinely stalls.

In [ ]:
BETA_MAX = 0.034
KL_WARMUP_EPOCHS = 50
EPOCHS = 400
LEARNING_RATE = 3e-4
MIN_LEARNING_RATE = 1e-6
EARLY_STOPPING_PATIENCE = 50
MIN_DELTA = 1e-4
GRADIENT_CLIP_NORM = 5.0
FIRST_DIFFERENCE_WEIGHT = 0.5
SECOND_DIFFERENCE_WEIGHT = 0.1
AREA_WEIGHT = 2.0
PEAK_WEIGHT = 1.0

def inverse_anscombe_tensor(x):
    transformed = x.clamp(0.0, 1.0) * TRANSFORM_SCALE
    return (((transformed / 2.0) ** 2) - 3.0 / 8.0).clamp_min(0.0)

def beta_vae_loss(reconstruction, target, mu, logvar, beta):
    point_loss = F.mse_loss(reconstruction, target, reduction="none").sum(dim=1).mean()
    rec_d1 = reconstruction[:, 1:] - reconstruction[:, :-1]
    target_d1 = target[:, 1:] - target[:, :-1]
    first_difference_loss = F.mse_loss(rec_d1, target_d1, reduction="none").sum(dim=1).mean()
    rec_d2 = rec_d1[:, 1:] - rec_d1[:, :-1]
    target_d2 = target_d1[:, 1:] - target_d1[:, :-1]
    second_difference_loss = F.mse_loss(rec_d2, target_d2, reduction="none").sum(dim=1).mean()

    raw_reconstruction = inverse_anscombe_tensor(reconstruction)
    raw_target = inverse_anscombe_tensor(target)
    target_area = raw_target.sum(dim=1).clamp_min(1.0)
    area_loss = (((raw_reconstruction.sum(dim=1) - target_area) / target_area) ** 2).mean()
    target_peak = raw_target.amax(dim=1).clamp_min(1.0)
    peak_loss = (((raw_reconstruction.amax(dim=1) - target_peak) / target_peak) ** 2).mean()

    reconstruction_loss = (
        point_loss
        + FIRST_DIFFERENCE_WEIGHT * first_difference_loss
        + SECOND_DIFFERENCE_WEIGHT * second_difference_loss
        + AREA_WEIGHT * area_loss
        + PEAK_WEIGHT * peak_loss
    )
    kl_loss = (-0.5 * (1 + logvar - mu.square() - logvar.exp())).sum(dim=1).mean()
    total_loss = reconstruction_loss + beta * kl_loss
    return total_loss, reconstruction_loss, kl_loss

@torch.no_grad()
def evaluate(model, loader, beta=BETA_MAX):
    model.eval()
    sums = np.zeros(3, dtype=np.float64)
    count = 0
    for x, target in loader:
        x = x.to(DEVICE, non_blocking=True)
        target = target.to(DEVICE, non_blocking=True)
        reconstruction, mu, logvar = model(x)
        values = beta_vae_loss(reconstruction, target, mu, logvar, beta)
        batch_size = x.shape[0]
        sums += np.asarray([v.item() for v in values]) * batch_size
        count += batch_size
    return sums / count

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-6)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=10, threshold=MIN_DELTA, min_lr=MIN_LEARNING_RATE
)
history = {"train_total": [], "train_reconstruction": [], "train_kl": [],
           "val_total": [], "val_reconstruction": [], "val_kl": [], "lr": [], "beta": []}
best_val = math.inf
best_state = None
epochs_without_improvement = 0
start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    beta = BETA_MAX * min(1.0, epoch / KL_WARMUP_EPOCHS)
    model.train()
    sums = np.zeros(3, dtype=np.float64)
    count = 0
    for x, target in train_loader:
        x = x.to(DEVICE, non_blocking=True)
        target = target.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        reconstruction, mu, logvar = model(x)
        loss, reconstruction_loss, kl_loss = beta_vae_loss(reconstruction, target, mu, logvar, beta)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_NORM)
        optimizer.step()
        batch_size = x.shape[0]
        sums += np.asarray([loss.item(), reconstruction_loss.item(), kl_loss.item()]) * batch_size
        count += batch_size

    train_values = sums / count
    val_values = evaluate(model, val_loader, beta=BETA_MAX)
    history["train_total"].append(train_values[0])
    history["train_reconstruction"].append(train_values[1])
    history["train_kl"].append(train_values[2])
    history["val_total"].append(val_values[0])
    history["val_reconstruction"].append(val_values[1])
    history["val_kl"].append(val_values[2])
    history["lr"].append(optimizer.param_groups[0]["lr"])
    history["beta"].append(beta)

    improved = val_values[0] < best_val - MIN_DELTA
    if improved:
        best_val = float(val_values[0])
        best_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
        torch.save({
            "model_state_dict": best_state,
            "input_dim": TARGET_LENGTH, "latent_dim": 12, "beta": BETA_MAX,
            "transform": "anscombe", "transform_scale": TRANSFORM_SCALE, "seed": SEED,
            "train_cases": sorted(train_cases),
            "validation_cases": sorted(val_cases),
            "test_cases": sorted(test_cases),
        }, CHECKPOINT_FILE)
    else:
        epochs_without_improvement += 1

    scheduler.step(val_values[0])
    if epoch == 1 or epoch % 10 == 0:
        print(
            f"epoch {epoch:03d} | train={train_values[0]:.4f} "
            f"(rec={train_values[1]:.4f}, kl={train_values[2]:.4f}) | "
            f"val={val_values[0]:.4f} | β={beta:.4f} | lr={history['lr'][-1]:.2e}"
        )
    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(f"Early stopping at epoch {epoch}.")
        break

if best_state is None:
    raise RuntimeError("Training produced no finite validation checkpoint.")
model.load_state_dict(best_state)
print(f"Best validation loss: {best_val:.4f}")
print(f"Training time: {(time.time() - start_time) / 60:.1f} minutes")
print(f"Checkpoint: {CHECKPOINT_FILE}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
epochs = np.arange(1, len(history["train_total"]) + 1)
for ax, key, title in zip(
    axes, ["total", "reconstruction", "kl"],
    ["Total β-VAE loss", "Reconstruction loss", "KL divergence"],
):
    ax.plot(epochs, history[f"train_{key}"], label="train")
    ax.plot(epochs, history[f"val_{key}"], label="validation")
    ax.set(title=title, xlabel="Epoch")
    ax.grid(alpha=0.25)
    ax.legend()
plt.tight_layout()
plt.show()

## Reconstruction and denoising check

The grey line is a noisy simulated acquisition, black is the deterministic VAE reconstruction obtained from the latent mean, and orange is PROPHESY's noise-free expected total signal. Training now maps the grey input to the orange target. Metrics are reported in both transformed space and raw-count space, including per-spectrum normalized RMSE, integrated-area error, peak-height error, and dominant-peak position error.

In [ ]:
@torch.no_grad()
def reconstruct_from_latent_mean(model, normalized_spectra, batch_size=1024):
    model.eval()
    reconstructed = []
    for start in range(0, len(normalized_spectra), batch_size):
        x = torch.from_numpy(normalized_spectra[start:start + batch_size]).to(DEVICE)
        mu, _ = model.encode(x)
        reconstructed.append(model.decode(mu).cpu().numpy())
    return np.concatenate(reconstructed)

test_reconstruction = reconstruct_from_latent_mean(model, x_all[test_idx])
raw_test_reconstruction = denormalize_counts(test_reconstruction)
raw_test_target = dataset["expected_total"][test_idx]
axis = dataset["axis"]

plot_rng = np.random.default_rng(SEED + 1)
example_positions = plot_rng.choice(len(test_idx), size=min(6, len(test_idx)), replace=False)
example_idx = test_idx[example_positions]
fig, axes = plt.subplots(len(example_idx), 1, figsize=(11, 2.5 * len(example_idx)), sharex=True)
axes = np.atleast_1d(axes)
for ax, sample_index, position in zip(axes, example_idx, example_positions):
    ax.plot(axis, dataset["noisy"][sample_index], color="0.70", lw=1.0, label="noisy input")
    ax.plot(axis, raw_test_reconstruction[position], color="black", lw=1.4, label="VAE reconstruction")
    ax.plot(axis, raw_test_target[position], color="tab:orange", lw=1.1, label="noise-free target")
    ax.set_ylabel("Counts")
    ax.grid(alpha=0.2)
axes[0].legend(ncol=3)
axes[-1].set_xlabel("Binding energy (eV)")
plt.tight_layout()
plt.show()

test_values = evaluate(model, test_loader)
transformed_mse = np.mean((test_reconstruction - target_all[test_idx]) ** 2)
raw_rmse = np.sqrt(np.mean((raw_test_reconstruction - raw_test_target) ** 2, axis=1))
raw_range = np.maximum(np.ptp(raw_test_target, axis=1), 1.0)
nrmse = raw_rmse / raw_range
target_area = np.maximum(raw_test_target.sum(axis=1), 1.0)
area_relative_error = np.abs(raw_test_reconstruction.sum(axis=1) - target_area) / target_area
target_peak = np.maximum(raw_test_target.max(axis=1), 1.0)
peak_relative_error = np.abs(raw_test_reconstruction.max(axis=1) - target_peak) / target_peak
predicted_peak_position = axis[np.argmax(raw_test_reconstruction, axis=1)]
target_peak_position = axis[np.argmax(raw_test_target, axis=1)]
peak_position_error = np.abs(predicted_peak_position - target_peak_position)

print(f"Test total={test_values[0]:.4f}, reconstruction={test_values[1]:.4f}, KL={test_values[2]:.4f}")
print(f"Anscombe-space MSE: {transformed_mse:.6f}")
print(f"Median per-spectrum NRMSE: {np.median(nrmse):.4f}")
print(f"Median integrated-area relative error: {np.median(area_relative_error):.2%}")
print(f"Median peak-height relative error: {np.median(peak_relative_error):.2%}")
print(f"Median dominant-peak position error: {np.median(peak_position_error):.4f} eV")

## Compare the latent space with known physical parameters

PROPHESY provides ground-truth peak centres and widths, while intensity and background summaries can be calculated directly. Repeated sweeps are averaged by physical spectrum before correlations are calculated, preventing configurations with many sweeps from dominating. Constant labels, such as fixed mixture fractions, are reported and excluded instead of generating invalid correlations. Highly collinear physical labels are also reported because a VAE cannot uniquely disentangle variables that never vary independently.

In [ ]:
@torch.no_grad()
def encode_means(model, normalized_spectra, batch_size=1024):
    model.eval()
    result = []
    for start in range(0, len(normalized_spectra), batch_size):
        x = torch.from_numpy(normalized_spectra[start:start + batch_size]).to(DEVICE)
        mu, _ = model.encode(x)
        result.append(mu.cpu().numpy())
    return np.concatenate(result)

z_test = encode_means(model, x_all[test_idx])
label_columns = {}
for i in range(3):
    label_columns[f"center {i + 1}"] = dataset["peak_center"][test_idx, i]
    label_columns[f"width {i + 1}"] = dataset["peak_width"][test_idx, i]
    label_columns[f"fraction {i + 1}"] = dataset["peak_fraction"][test_idx, i]
label_columns["photon energy"] = dataset["photon_energy"][test_idx]
label_columns["log total area"] = np.log1p(dataset["expected_total"][test_idx].sum(axis=1))
label_columns["log background area"] = np.log1p(dataset["background"][test_idx].sum(axis=1))
label_columns["log peak height"] = np.log1p(dataset["expected_total"][test_idx].max(axis=1))

physical_names_all = list(label_columns)
physical_all = np.column_stack([label_columns[name] for name in physical_names_all])
test_sources = dataset["source"][test_idx]
unique_test_sources = np.unique(test_sources)
z_by_source = []
physical_by_source = []
for source in unique_test_sources:
    mask = test_sources == source
    z_by_source.append(z_test[mask].mean(axis=0))
    physical_by_source.append(physical_all[mask].mean(axis=0))
z_by_source = np.asarray(z_by_source)
physical_by_source = np.asarray(physical_by_source)

varying = np.isfinite(physical_by_source).all(axis=0) & (physical_by_source.std(axis=0) > 1e-8)
excluded_names = [name for name, keep in zip(physical_names_all, varying) if not keep]
physical_names = [name for name, keep in zip(physical_names_all, varying) if keep]
physical = physical_by_source[:, varying]
print(f"Physical spectra used for correlation: {len(unique_test_sources)}")
print(f"Excluded constant/non-finite labels: {excluded_names or 'none'}")

correlation = np.empty((z_by_source.shape[1], physical.shape[1]), dtype=np.float64)
for i in range(z_by_source.shape[1]):
    for j in range(physical.shape[1]):
        correlation[i, j] = np.corrcoef(z_by_source[:, i], physical[:, j])[0, 1]

label_correlation = np.corrcoef(physical, rowvar=False)
for i in range(len(physical_names)):
    for j in range(i + 1, len(physical_names)):
        if abs(label_correlation[i, j]) >= 0.98:
            print(f"Collinear labels: {physical_names[i]} and {physical_names[j]} (r={label_correlation[i, j]:+.3f})")

fig, ax = plt.subplots(figsize=(max(10, 0.9 * len(physical_names)), 6))
image = ax.imshow(correlation, vmin=-1, vmax=1, cmap="coolwarm", aspect="auto")
ax.set_xticks(np.arange(len(physical_names)), physical_names, rotation=45, ha="right")
ax.set_yticks(np.arange(12), [f"z{i}" for i in range(12)])
ax.set_title("Pearson correlation: latent means vs. varying PROPHESY properties")
fig.colorbar(image, ax=ax, label="Correlation")
plt.tight_layout()
plt.show()

for j, name in enumerate(physical_names):
    best = int(np.argmax(np.abs(correlation[:, j])))
    print(f"{name:>20}: z{best} (r={correlation[best, j]:+.3f})")

## Generate artificial spectra from the prior

These are unconditional samples from `z ~ N(0, I)`. Before plotting them, the cell reports the aggregated encoder mean and standard deviation for every latent dimension. Values close to zero and one respectively indicate that the learned posterior is compatible with the sampling prior; large departures warn that unconditional generations should not yet be used as scientific data.

In [ ]:
N_GENERATED = 12
train_latent_means = encode_means(model, x_all[train_idx])
aggregate_mean = train_latent_means.mean(axis=0)
aggregate_std = train_latent_means.std(axis=0)
print("Aggregated latent mean:", np.round(aggregate_mean, 3))
print("Aggregated latent std: ", np.round(aggregate_std, 3))
print(f"Largest |mean|={np.max(np.abs(aggregate_mean)):.3f}; largest |std-1|={np.max(np.abs(aggregate_std - 1.0)):.3f}")

model.eval()
with torch.no_grad():
    z = torch.randn(N_GENERATED, 12, device=DEVICE)
    generated_normalized = model.decode(z).cpu().numpy()
generated_counts = denormalize_counts(generated_normalized)

fig, axes = plt.subplots(3, 4, figsize=(14, 8), sharex=True)
for ax, spectrum in zip(axes.flat, generated_counts):
    ax.plot(axis, spectrum, color="tab:blue", lw=1.2)
    ax.grid(alpha=0.2)
for ax in axes[-1]:
    ax.set_xlabel("Binding energy (eV)")
for ax in axes[:, 0]:
    ax.set_ylabel("Counts")
fig.suptitle("β-VAE prior samples", y=1.01)
plt.tight_layout()
plt.show()

## Interpretation and next checks

A successful run should show: (1) smooth reconstructions that retain the simulated C1s components, (2) low raw-count area and peak-height errors against `Stot`, (3) similar validation and test metrics after the balanced group split, and (4) aggregate latent means near zero and standard deviations near one before standard-normal prior samples are trusted.

The model architecture remains the paper-style fully connected β-VAE. Accuracy improvements come from the PROPHESY-specific training protocol: noise-free reconstruction targets, variance-stabilized counts, balanced physical-spectrum sampling, group-balanced splits, shape/area/peak losses, KL warm-up, adaptive learning rate, and raw-unit evaluation. Constant or collinear simulation parameters still cannot become independently disentangled; obtaining that scientific result requires generating new PROPHESY cases in which those parameters vary independently.